## Overview

Data validation is how we prevent bad data from silently becoming bad analysis.

---

## Learning Objectives


* Apply data-quality checks and assertions to validate datasets
* Detect and handle errors, anomalies, and inconsistencies in source data
* Design validation rules aligned with analytical goals
* Generate and document a reproducible data-quality report for project files
* Communicate data-quality findings clearly and transparently

In [1]:
# import pandas, numpy

import pandas as pd
import numpy as np

---

We're going to use `numpy` and `pandas` to simulate a dataset that needs to be validated. 

Let's first discuss what is happening with the following code:

```
np.random.seed(42)
```

In [3]:
np.random.seed(42)

### What's happening with this and why do we do this?

`np.random.seed` initializes the pseudo-random number generator in NumPy by setting its starting point (the seed). This makes "random" processes deterministic, ensuring that you get the exact same sequence of numbers every time the code is run with that same seed. 

### Key Functions and Benefits
* Reproducibility: It allows others to get the same results as you, which is critical for scientific experiments and sharing code.
* Debugging: By making random behavior predictable, you can more easily find and fix bugs that might otherwise only occur sporadically.
* Consistency in ML: In machine learning, it ensures that data splits (train/test) or initial weight initializations remain the same across different training runs.

In [7]:
np.random.seed(42)

np.random.choice([1, 2, 3, 4, 5, 6])

4

---

Now let's simulate our dataset.

In [9]:
# number of observations
n = 200

In [11]:
# Base clean data
# Base clean data
sales = pd.DataFrame({
    "order_id": np.arange(1000, 1000+n),
    "customer_id": np.random.randint(1, 50, size=n),
    "revenue": np.round(np.random.normal(loc=200, scale=75, size=n), 2),
    "quantity": np.random.randint(1, 5, size=n),
    "status": np.random.choice(
        ["Complete", "Cancelled", "Returned"],
        size=n,
        p=[0.7, 0.2, 0.1]
    )
})

sales.head()

,order_id,customer_id,revenue,quantity,status
0,1000,29,43.13,1,Complete
1,1001,15,294.78,3,Complete
2,1002,43,198.84,4,Complete
3,1003,8,197.95,4,Complete
4,1004,21,261.34,3,Complete


Inject Realistic Data Issues

In [ ]:
# 1. Duplicate Order ID's


In [ ]:
# 2. Negative Revenue


In [ ]:
# 3. Extreme outliers


In [ ]:
# 4. Invalid Quantities


In [ ]:
# 5. Missing Values


In [ ]:
# 6. Inconsistent formatting


In [13]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 200

# Base clean data
sales = pd.DataFrame({
    "order_id": np.arange(1000, 1000 + n),
    "customer_id": np.random.randint(1, 50, size=n),
    "revenue": np.round(np.random.normal(loc=200, scale=75, size=n), 2),
    "quantity": np.random.randint(1, 5, size=n),
    "status": np.random.choice(
        ["Complete", "Cancelled", "Returned"],
        size=n,
        p=[0.7, 0.2, 0.1]
    )
})

# -----------------------------
# Inject REALISTIC DATA ISSUES
# -----------------------------

# 1. Duplicate order IDs
duplicate_indices = np.random.choice(sales.index, size=10, replace=False)
sales.loc[duplicate_indices, "order_id"] = sales.loc[duplicate_indices, "order_id"].values - 1

# 2. Negative revenue
neg_indices = np.random.choice(sales.index, size=8, replace=False)
sales.loc[neg_indices, "revenue"] = -abs(sales.loc[neg_indices, "revenue"])

# 3. Extreme outliers
outlier_indices = np.random.choice(sales.index, size=5, replace=False)
sales.loc[outlier_indices, "revenue"] = sales.loc[outlier_indices, "revenue"] * 20

# 4. Invalid quantities
bad_qty_indices = np.random.choice(sales.index, size=6, replace=False)
sales.loc[bad_qty_indices, "quantity"] = 0

# 5. Missing values
missing_indices = np.random.choice(sales.index, size=10, replace=False)
sales.loc[missing_indices, "customer_id"] = np.nan

# 6. Inconsistent status formatting
status_noise = ["complete", " COMPLETE ", "cancelled", "returned", "CANCELLED"]
status_indices = np.random.choice(sales.index, size=15, replace=False)
sales.loc[status_indices, "status"] = np.random.choice(status_noise, size=15)

sales.head()

,order_id,customer_id,revenue,quantity,status
0,1000,39.0,341.46,2,Complete
1,1001,29.0,213.09,1,Complete
2,1002,15.0,-219.32,3,Complete
3,1003,43.0,194.42,4,Complete
4,1004,8.0,56.09,4,Complete


---

## Preview our data

Let's check out `.head()`, `.info()`, and `.describe()`

and our `status` column

In [17]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     200 non-null    int32  
 1   customer_id  190 non-null    float64
 2   revenue      200 non-null    float64
 3   quantity     200 non-null    int32  
 4   status       200 non-null    object 
dtypes: float64(2), int32(2), object(1)
memory usage: 6.4+ KB


In [15]:
sales.describe()

,order_id,customer_id,revenue,quantity
count,200.000000,190.000000,200.000000,200.00000
mean,1099.450000,25.157895,249.764850,2.36000
std,57.871696,14.125526,773.012367,1.21977
min,1000.000000,1.000000,-5633.400000,0.00000
25%,1049.750000,13.000000,140.122500,1.00000
50%,1099.500000,25.500000,204.670000,2.00000
75%,1149.250000,37.750000,249.345000,3.00000
max,1199.000000,49.000000,5240.800000,4.00000


In [12]:
sales['status'].value_counts()

status
Complete      138
Cancelled      29
Returned       18
 COMPLETE       6
complete        3
returned        3
CANCELLED       2
cancelled       1
Name: count, dtype: int64

---

## Why Validation Matters

Let's investigate the revenue column

In [20]:
sales["revenue"].sum()

49952.97

In [22]:
sales["revenue"].mean()

249.76485

In [26]:
sales.groupby("status")["revenue"].sum()

status
 COMPLETE      1401.25
CANCELLED       307.34
Cancelled      -619.58
Complete      44895.30
Returned       2841.77
cancelled       157.53
complete        724.09
returned        245.27
Name: revenue, dtype: float64

## Check for missing values

In [30]:
sales.isna().sum()

order_id        0
customer_id    10
revenue         0
quantity        0
status          0
dtype: int64

## Check for Duplicate ID's

In [32]:
sales[sales["customer_id"].isna()]

,order_id,customer_id,revenue,quantity,status
15,1014,NaN,131.80,3,Cancelled
46,1046,NaN,149.00,1,Complete
66,1066,NaN,194.22,3,Cancelled
67,1067,NaN,225.59,2,Complete
77,1077,NaN,236.19,1,Complete
80,1080,NaN,235.49,4,Complete
88,1088,NaN,212.99,2,Complete
141,1141,NaN,151.00,3,Complete
182,1182,NaN,221.07,1,Complete
194,1194,NaN,137.83,4,Complete


## Check for invalid numeric values

In [46]:
sales["order_id"].duplicated().sum()

10

In [38]:
sales[sales["order_id"].duplicated(keep=False)].sort_values(by='order_id')

,order_id,customer_id,revenue,quantity,status
14,1014,24.0,-259.33,1,Returned
15,1014,NaN,131.80,3,Cancelled
19,1019,44.0,364.28,1,Complete
20,1019,30.0,125.71,2,Returned
39,1039,7.0,258.64,3,complete
40,1039,21.0,107.23,4,Returned
46,1046,NaN,149.00,1,Complete
47,1046,9.0,217.42,1,Complete
97,1096,7.0,279.04,2,Complete
96,1096,25.0,281.23,1,Complete


In [42]:
sales['order_id'].is_unique

False

In [48]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     200 non-null    int32  
 1   customer_id  190 non-null    float64
 2   revenue      200 non-null    float64
 3   quantity     200 non-null    int32  
 4   status       200 non-null    object 
dtypes: float64(2), int32(2), object(1)
memory usage: 6.4+ KB


In [50]:
sales[sales["revenue"]<= 0]

,order_id,customer_id,revenue,quantity,status
2,1002,15.0,-219.32,3,Complete
14,1014,24.0,-259.33,1,Returned
30,1030,27.0,-141.26,4,Returned
82,1082,21.0,-136.49,1,returned
106,1106,8.0,-271.55,3,Complete
119,1119,34.0,-5633.40,3,Cancelled
137,1137,1.0,-264.93,3,Complete
156,1156,23.0,-43.10,4,Cancelled
163,1163,13.0,-209.81,4,Complete


In [52]:
sales[sales["quantity"]<= 0]

,order_id,customer_id,revenue,quantity,status
42,1042,39.0,239.15,0,Cancelled
53,1053,7.0,249.24,0,Complete
83,1083,16.0,86.39,0,Returned
115,1115,7.0,59.96,0,Complete
151,1151,49.0,146.92,0,Complete
188,1188,39.0,226.78,0,returned


## Check for outliers

In [55]:
sales.describe()

,order_id,customer_id,revenue,quantity
count,200.000000,190.000000,200.000000,200.00000
mean,1099.450000,25.157895,249.764850,2.36000
std,57.871696,14.125526,773.012367,1.21977
min,1000.000000,1.000000,-5633.400000,0.00000
25%,1049.750000,13.000000,140.122500,1.00000
50%,1099.500000,25.500000,204.670000,2.00000
75%,1149.250000,37.750000,249.345000,3.00000
max,1199.000000,49.000000,5240.800000,4.00000


## A Rules-based outlier check

In [57]:
#sales.describe()

q1 = sales["revenue"].quantile(0.25)
q3 = sales["revenue"].quantile(0.75)

iqr = q3 - q1

In [61]:
lower_bound = q1 - 1.5*iqr
upper_bound = q3 + 1.5*iqr

In [63]:
revenue_outliers = sales[
    (sales["revenue"] < lower_bound) | 
    (sales["revenue"] > upper_bound )

]
revenue_outliers

,order_id,customer_id,revenue,quantity,status
2,1002,15.0,-219.32,3,Complete
14,1014,24.0,-259.33,1,Returned
22,1022,2.0,4149.40,4,Complete
27,1027,44.0,4710.40,3,Complete
30,1030,27.0,-141.26,4,Returned
69,1069,26.0,5240.80,3,Complete
82,1082,21.0,-136.49,1,returned
103,1103,24.0,488.95,2,Complete
104,1104,11.0,4856.40,4,Complete
106,1106,8.0,-271.55,3,Complete


## Check Category Consistency

In [65]:
sales["status"].value_counts()

status
Complete      138
Cancelled      29
Returned       18
 COMPLETE       6
complete        3
returned        3
CANCELLED       2
cancelled       1
Name: count, dtype: int64

## Clean and Validate Categories

In [67]:
sales["status_clean"] = sales["status"].str.lower().str.strip()
sales["status_clean"].value_counts()

status_clean
complete     147
cancelled     32
returned      21
Name: count, dtype: int64

## Check Date Validity

---

So we've seen how to validate our data, now we'll add what are known as validation flags.

Instead of immediately deleting rows, we create columns that identify specific problems.

This let's us answer:
* Which rows have problems?
* What kind of problems?
* How many problems exist?
* Are some rows affected by multiple issues?

---

## Creating Validation Flags

In [69]:
sales["missing_customer_id"] =sales["customer_id"].isna()

In [71]:
sales.head()

,order_id,customer_id,revenue,quantity,status,status_clean,missing_customer_id
0,1000,39.0,341.46,2,Complete,complete,False
1,1001,29.0,213.09,1,Complete,complete,False
2,1002,15.0,-219.32,3,Complete,complete,False
3,1003,43.0,194.42,4,Complete,complete,False
4,1004,8.0,56.09,4,Complete,complete,False


Now that we have our validation flags, we can count our validation issues.

---

## Count Validation Issues

In [73]:
valid_statuses = ["complete", "cancelled","returned"]

In [75]:
sales["duplicate_order_id"] = sales["order_id"].duplicated(keep=False)
sales["negative_revenue"] = sales["revenue"] < 0
sales["invalid_quantity"] = sales["quantity"] <= 0
sales["invalid_status"] = ~sales["status_clean"].isin(valid_statuses)

sales.head()

,order_id,customer_id,revenue,quantity,status,status_clean,missing_customer_id,duplicate_order_id,negative_revenue,invalid_quantity,invalid_status
0,1000,39.0,341.46,2,Complete,complete,False,False,False,False,False
1,1001,29.0,213.09,1,Complete,complete,False,False,False,False,False
2,1002,15.0,-219.32,3,Complete,complete,False,False,True,False,False
3,1003,43.0,194.42,4,Complete,complete,False,False,False,False,False
4,1004,8.0,56.09,4,Complete,complete,False,False,False,False,False


In [77]:
validation_columns = [


    "missing_customer_id",
    "duplicate_order_id",
    "negative_revenue",
    "invalid_quantity",
    "invalid_status"
    
]

## Identify rows with any issue

In [79]:
#sales["has_quality_issue"] = sales[validation_columns]
sales[validation_columns].sum()

sales["has_quality_issue"] = sales[validation_columns].any(axis=1)

problem_rows = sales[sales['has_quality_issue']]

problem_rows.head()

,order_id,customer_id,revenue,quantity,status,status_clean,missing_customer_id,duplicate_order_id,negative_revenue,invalid_quantity,invalid_status,has_quality_issue
2,1002,15.0,-219.32,3,Complete,complete,False,False,True,False,False,True
14,1014,24.0,-259.33,1,Returned,returned,False,True,True,False,False,True
15,1014,NaN,131.80,3,Cancelled,cancelled,True,True,False,False,False,True
19,1019,44.0,364.28,1,Complete,complete,False,True,False,False,False,True
20,1019,30.0,125.71,2,Returned,returned,False,True,False,False,False,True


In [ ]:
#analysis_ready_sales= sales[~sales]["has_quality_issue"]].copy()

## Now compare results before and after validation

In [81]:
raw_revenue_total = sales["revenue"].sum()
raw_avg_order_value = sales["revenue"].mean()

print(raw_revenue_total, raw_avg_order_value)

analysis_ready_sales = sales[~sales["has_quality_issue"]].copy()

clean_revenue_total = analysis_ready_sales["revenue"].sum()
clean_avg_order_value = analysis_ready_sales["revenue"].mean()

clean_revenue_total, clean_avg_order_value

49952.97 249.76485


(50572.70999999999, 320.08044303797465)

---

## Assertions

Assertions are useful when a rule must be true before analysis continues.

Python has the built-in `assert` that *asserts* a condition is true or raises an `AssertionError`. 

Here's a quick example:

In [83]:
assert sales["order_id"].is_unique,"Order ID's must be unique."

AssertionError: Order ID's must be unique.

## Assert our different columns for validation

In [85]:
assert sales["customer_id"].notna().all()," Customer ID cannot be missing "

AssertionError:  Customer ID cannot be missing 

## Now let's build a validation function with `assert`

In [89]:
def validate_sales_data(df):
    assert df ["order_id"].is_unique, "Order ID Msut be unique!"
    assert df["customer_id"].notna().all, "Customer ID cannot be missing"
    assert (df["revenue"] >= 0).all(), "Revenue cannnopt be negative"
    assert (df["quantity"] > 0 ).all(), " quantity must be greater than zero "
    cleaned_status = df["status"].str.lower().str.strip()
    valid_statuses = ["complete", "cancelled", "returned"]

    assert cleaned_status.isin(valid_statuses).all(), "Status contains an invalid status"

    return True

validate_sales_data(sales)

#asserting if these statementsg are tru raises error if they are not validated 

AssertionError: Order ID Msut be unique!

In [91]:
def validate_sales_data1(df):
    assert df["order_id"].is_unique, "Order ID must be unique"
    assert df["customer_id"].notna().all(), "Customer ID cannot be missing"
    assert (df['revenue'] >= 0).all(), "Revenue cannot be negative"
    assert (df["quantity"] > 0).all(), "Quantity must be greater than zero"

    cleaned_status = df["status"].str.lower().str.strip()
    valid_statuses = ["complete", "cancelled", "returned"]

    assert cleaned_status.isin(valid_statuses).all(), "Status contains an invalid status"

    return True

validate_sales_data1(sales)

AssertionError: Order ID must be unique

## Build a validation report

In [97]:
analysis_ready_sales = sales[~sales["has_quality_issue"]].copy()

analysis_ready_sales

,order_id,customer_id,revenue,quantity,status,status_clean,missing_customer_id,duplicate_order_id,negative_revenue,invalid_quantity,invalid_status,has_quality_issue
0,1000,39.0,341.46,2,Complete,complete,False,False,False,False,False,False
1,1001,29.0,213.09,1,Complete,complete,False,False,False,False,False,False
3,1003,43.0,194.42,4,Complete,complete,False,False,False,False,False,False
4,1004,8.0,56.09,4,Complete,complete,False,False,False,False,False,False
5,1005,21.0,198.01,3,Complete,complete,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
192,1192,9.0,260.96,4,Returned,returned,False,False,False,False,False,False
193,1193,37.0,247.22,4,Complete,complete,False,False,False,False,False,False
195,1195,42.0,157.99,2,Complete,complete,False,False,False,False,False,False
198,1198,15.0,198.43,4,Complete,complete,False,False,False,False,False,False


---

We can make the report easier to read with some control flow.

In [99]:
analysis_ready_sales.groupby("status_clean")['revenue'].sum()

status_clean
cancelled     4554.22
complete     43346.43
returned      2672.06
Name: revenue, dtype: float64

In [101]:
analysis_ready_sales.groupby("status_clean")['revenue'].mean()

status_clean
cancelled    182.168800
complete     358.234959
returned     222.671667
Name: revenue, dtype: float64

---

Let's create an analysis ready dataset!

Then validate it

---

## Final Business Analysis

---

## Wrap-Up


* Raw data can silently produce misleading answers.
* Pandas makes validation practical.
* Boolean masks help identify bad rows.
* Assertions enforce must-pass rules.
* Validation reports summarize data quality.
* Clean datasets should be created from explicit rules, not vibes.

### Final Takeaway:

**Data validation is the checkpoint between "I have data" and "I trust this analysis".**